<a href="https://colab.research.google.com/github/Wapikkk/text-summarize-nlp/blob/extractive/02_baseline_leadk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **02 - Baseline Lead-k**

**Tujuan**: menyediakan baseline sederhana, yaitu k kalimat pertama dokumen, sebagai titik acuan bagi model abstractive (GRU+attention, mT5, IndoBART).

Dalam hal ini, Lead-k bukan sebagai model model. Melainkan hanya menyalin kalimat awal.

**Output**: results/extractive_lead_k2.jsonl (kolom id, reference, prediction) dan waktu proses di extractive_run_info.json.


## **Install Package**
Memasang library untuk menghitung ROUGE.

In [ ]:
!pip -q install rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## **Import Package**
Mengimpor library yang dipakai, menghubungkan Google Drive, dan mendefinisikan `read_jsonl` untuk membaca file split.

In [ ]:
import re, json, os, time
import numpy as np

from google.colab import drive
from rouge_score import rouge_scorer

drive.mount('/content/drive')

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

Mounted at /content/drive


## **Load Dataset**
Membaca `val.jsonl` dan `test.jsonl` hasil 01_data dari folder Drive (`dokumen_nlp`). Val dipakai untuk memilih `k`, dan test hanya untuk hasil akhir. Lead-k tidak memerlukan data train.

In [ ]:
DATA_DIR = '/content/drive/MyDrive/dokumen_nlp'  # direktori hasil preprocessing 01_data

val  = read_jsonl(f'{DATA_DIR}/val.jsonl')
test = read_jsonl(f'{DATA_DIR}/test.jsonl')

print('Informasi hasil preprocessing 01_data:')
print('  val  :', len(val))
print('  test :', len(test))

Informasi hasil preprocessing 01_data:
  train: 37970
  val  : 4725
  test : 4778


## **Pemisahan Kalimat**
Dokumen dipecah menjadi kalimat dengan regex: pemisah adalah tanda `. ! ?` yang diikuti huruf kapital, angka, atau tanda kutip. Kalimat kurang dari 3 kata dibuang. Di data val, jumlah kalimat per dokumen adalah median 21 (p10 = 8, p90 = 42), dan panjang ringkasan referensi median 27 kata (p10 = 13, p90 = 54). Ini menjadi dasar memilih jumlah kalimat `k` yang kecil.

In [ ]:
def split_sentences(text):
    text = re.sub(r'\s+', ' ', text).strip()
    sents = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"“(])', text)
    sents = [s for s in sents if len(s.split()) >= 3]
    return sents or [text]

n_sent  = [len(split_sentences(r['document'])) for r in val]
ref_len = [len(r['summary'].split()) for r in val]
print('kalimat/dokumen (p10, p50, p90):', np.percentile(n_sent,  [10, 50, 90]))
print('kata/ringkasan  (p10, p50, p90):', np.percentile(ref_len, [10, 50, 90]))

kalimat/dokumen (p10, p50, p90): [ 8. 21. 42.]
kata/ringkasan  (p10, p50, p90): [13. 27. 54.]


## **Baseline Lead-k**
Lead-k mengambil `k` kalimat pertama dokumen sebagai ringkasan. Tidak ada pelatihan dan tidak ada penilaian kalimat. Hasilnya menjadi titik acuan: model abstractive harus mengungguli skor ini agar dianggap benar-benar belajar meringkas.

In [ ]:
def lead_summ(doc, k):
    return ' '.join(split_sentences(doc)[:k])

## **Contoh Hasil Ringkasan**
Satu dokumen val diringkas dengan Lead-2 dan dibandingkan dengan ringkasan referensi, untuk pemeriksaan kasat mata.

In [ ]:
doc = val[0]['document']
print('DOKUMEN  :', doc[:400], '\n')
print('REFERENSI:', val[0]['summary'], '\n')
print('LEAD-2   :', lead_summ(doc, 2))

## **Fungsi Evaluasi ROUGE**
Menghitung ROUGE-1, ROUGE-2, dan ROUGE-L (F1) dengan `rouge_score`, lalu memberikan rata-rata di seluruh contoh. Perhitungan akhir dilakukan di 04_evaluation dengan skrip bersama.

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'])

def mean_rouge(preds, refs):
    out = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        for m in out:
            out[m].append(s[m].fmeasure)
    return {m: round(float(np.mean(v)), 4) for m, v in out.items()}

DOKUMEN  : Di musim tertentu, buah dan sayuran dijual dengan harga yang lebih murah. Sering kali, swalayan memberi diskon khusus untuk musim tertentu karena para pembeli mencari buah dan sayuran yang spesifik pada waktu tersebut. Sebagai bonus, saat sedang musimnya, buah dan sayuran yang dijual terasa lebih lezat. Di musim gugur, carilah produk seperti labu, squash musim dingin, kol, dan apel. Di musim dingi 

REFERENSI: Belilah bahan makanan sesuai musimnya. Periksalah harga makanan beku dan kalengan. Periksa diskon mingguan. Selalu bandingkan harga jual. Belilah dalam jumlah besar. Gunakanlah kupon. Di Amerika Serikat, Anda dapat mendaftarkan diri untuk program SNAP atau WIC. Carilah pasar atau peternakan terdekat. 

TFIDF : Di musim dingin, carilah squash musim dingin, kol, serta umbi-umbian lainnya. Manfaatkanlah kupon jika Anda memilikinya. 

TEXTRANK : Di musim tertentu, buah dan sayuran dijual dengan harga yang lebih murah. Selain itu, karena ada banyak produk dijual murah di mu

## **Memilih k di Data Val**
Lead-k dicoba dengan `k` = 1 sampai 4 pada 1.000 contoh val. ROUGE naik seiring ringkasan memanjang, tetapi `k = 4` menghasilkan ringkasan sekitar 60 kata, hampir dua kali panjang referensi. Agar adil bagi model abstractive, dipilih **`k = 2`** (sekitar 30 kata, setara panjang referensi).

In [ ]:
val_s = val[:1000]
refs = [r['summary'] for r in val_s]
print('rata-rata panjang referensi (kata):', round(np.mean([len(x.split()) for x in refs]), 1))

for k in [1, 2, 3, 4]:
    preds = [lead_summ(r['document'], k) for r in val_s]
    print('Lead-', k, '|', mean_rouge(preds, refs),
          '| panjang rata-rata:', round(np.mean([len(p.split()) for p in preds]), 1))

BASELINE_K = 2  # dipilih karena panjangnya paling dekat dengan referensi

## **Menyimpan Hasil Prediksi Test**
Lead-2 dijalankan pada seluruh data test dan disimpan sebagai JSONL (`id`, `reference`, `prediction`) di `dokumen_nlp/results/`, beserta waktu prosesnya di `extractive_run_info.json`.

In [ ]:
RESULT_DIR = f'{DATA_DIR}/results'
os.makedirs(RESULT_DIR, exist_ok=True)

t0 = time.time()
preds = [lead_summ(r['document'], BASELINE_K) for r in test]
dt = time.time() - t0

with open(f'{RESULT_DIR}/extractive_lead_k2.jsonl', 'w', encoding='utf-8') as f:
    for r, p in zip(test, preds):
        f.write(json.dumps({'id': r['id'], 'reference': r['summary'], 'prediction': p},
                           ensure_ascii=False) + '\n')

with open(f'{RESULT_DIR}/extractive_run_info.json', 'w') as f:
    json.dump({'lead_k2': {'k': BASELINE_K, 'infer_time_s': round(dt, 1), 'n_test': len(test)}}, f, indent=2)

print('Lead-2 selesai |', round(dt, 1), 'detik |', len(preds), 'ringkasan')

rata-rata panjang referensi (kata): 32.7
tfidf | k = 1 | {'rouge1': 0.1307, 'rouge2': 0.0288, 'rougeL': 0.1072, 'len': np.float64(16.2)}
tfidf | k = 2 | {'rouge1': 0.1809, 'rouge2': 0.0408, 'rougeL': 0.1369, 'len': np.float64(32.6)}
tfidf | k = 3 | {'rouge1': 0.2011, 'rouge2': 0.0478, 'rougeL': 0.1466, 'len': np.float64(48.8)}
tfidf | k = 4 | {'rouge1': 0.2046, 'rouge2': 0.0503, 'rougeL': 0.1462, 'len': np.float64(64.3)}
textrank | k = 1 | {'rouge1': 0.1969, 'rouge2': 0.0511, 'rougeL': 0.1514, 'len': np.float64(18.0)}
textrank | k = 2 | {'rouge1': 0.2331, 'rouge2': 0.0628, 'rougeL': 0.1699, 'len': np.float64(35.0)}
textrank | k = 3 | {'rouge1': 0.2355, 'rouge2': 0.0646, 'rougeL': 0.1687, 'len': np.float64(51.4)}
textrank | k = 4 | {'rouge1': 0.2299, 'rouge2': 0.0648, 'rougeL': 0.1632, 'len': np.float64(67.3)}
lsa | k = 1 | {'rouge1': 0.1481, 'rouge2': 0.0349, 'rougeL': 0.1176, 'len': np.float64(14.7)}
lsa | k = 2 | {'rouge1': 0.1888, 'rouge2': 0.0441, 'rougeL': 0.1434, 'len': np.float6